In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup


In [3]:
url = "https://books.toscrape.com/catalogue/page-1.html"

response = requests.get(
    url,
    headers={"User-Agent": "Nina Data Engineering Practice"},
    timeout=30
)


In [4]:
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

books = soup.select("article.product_pod")

print("Books found:", len(books))

#requests.get() retrieves the page’s HTML. BeautifulSoup parses that HTML, while select() finds elements matching the supplied CSS selector.

Books found: 20


In [5]:
#Extracting the records.Each dictionary becomes one DataFrame row.

records = []

for book in books:
    title = book.select_one("h3 a")["title"]

    price = book.select_one(".price_color").get_text(
        strip=True
    )

    availability = book.select_one(
        ".availability"
    ).get_text(strip=True)

    rating_classes = book.select_one(
        ".star-rating"
    ).get("class")

    rating = rating_classes[1]

    records.append({
        "title": title,
        "price_raw": price,
        "availability": availability,
        "rating": rating,
        "source_url": url
    })

books_df = pd.DataFrame(records)

print(books_df.head())
print(books_df.shape)

                                   title price_raw availability rating  \
0                   A Light in the Attic   Â£51.77     In stock  Three   
1                     Tipping the Velvet   Â£53.74     In stock    One   
2                             Soumission   Â£50.10     In stock    One   
3                          Sharp Objects   Â£47.82     In stock   Four   
4  Sapiens: A Brief History of Humankind   Â£54.23     In stock   Five   

                                         source_url  
0  https://books.toscrape.com/catalogue/page-1.html  
1  https://books.toscrape.com/catalogue/page-1.html  
2  https://books.toscrape.com/catalogue/page-1.html  
3  https://books.toscrape.com/catalogue/page-1.html  
4  https://books.toscrape.com/catalogue/page-1.html  
(20, 5)


In [6]:
#scrape 3 pages
records = []

for book in books:
    title = book.select_one("h3 a")["title"]

    price = book.select_one(".price_color").get_text(
        strip=True
    )

    availability = book.select_one(
        ".availability"
    ).get_text(strip=True)

    rating_classes = book.select_one(
        ".star-rating"
    ).get("class")

    rating = rating_classes[1]

    records.append({
        "title": title,
        "price_raw": price,
        "availability": availability,
        "rating": rating,
        "source_url": url
    })

books_df = pd.DataFrame(records)

print(books_df.head())
print(books_df.shape)

                                   title price_raw availability rating  \
0                   A Light in the Attic   Â£51.77     In stock  Three   
1                     Tipping the Velvet   Â£53.74     In stock    One   
2                             Soumission   Â£50.10     In stock    One   
3                          Sharp Objects   Â£47.82     In stock   Four   
4  Sapiens: A Brief History of Humankind   Â£54.23     In stock   Five   

                                         source_url  
0  https://books.toscrape.com/catalogue/page-1.html  
1  https://books.toscrape.com/catalogue/page-1.html  
2  https://books.toscrape.com/catalogue/page-1.html  
3  https://books.toscrape.com/catalogue/page-1.html  
4  https://books.toscrape.com/catalogue/page-1.html  
(20, 5)


In [8]:
import time

all_records = []

for page_number in range(1, 6):
    page_url = (
        "https://books.toscrape.com/catalogue/"
        f"page-{page_number}.html"
    )

    response = requests.get(
        page_url,
        headers={"User-Agent": "Nina Data Engineering Practice"},
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.select("article.product_pod")

    for book in books:
        rating_classes = book.select_one(
            ".star-rating"
        ).get("class")

        all_records.append({
            "title": book.select_one("h3 a")["title"],
            "price_raw": book.select_one(
                ".price_color"
            ).get_text(strip=True),
            "availability": book.select_one(
                ".availability"
            ).get_text(strip=True),
            "rating": rating_classes[1],
            "page_number": page_number,
            "source_url": page_url
        })

    print(
        f"Page {page_number}: {len(books)} books extracted"
    )

    time.sleep(1)

books_df = pd.DataFrame(all_records)

print("Total records:", len(books_df))
print(books_df.head())

Page 1: 20 books extracted
Page 2: 20 books extracted
Page 3: 20 books extracted
Page 4: 20 books extracted
Page 5: 20 books extracted
Total records: 100
                                   title price_raw availability rating  \
0                   A Light in the Attic   Â£51.77     In stock  Three   
1                     Tipping the Velvet   Â£53.74     In stock    One   
2                             Soumission   Â£50.10     In stock    One   
3                          Sharp Objects   Â£47.82     In stock   Four   
4  Sapiens: A Brief History of Humankind   Â£54.23     In stock   Five   

   page_number                                        source_url  
0            1  https://books.toscrape.com/catalogue/page-1.html  
1            1  https://books.toscrape.com/catalogue/page-1.html  
2            1  https://books.toscrape.com/catalogue/page-1.html  
3            1  https://books.toscrape.com/catalogue/page-1.html  
4            1  https://books.toscrape.com/catalogue/page-1.html  

In [13]:
books_df["price_numeric"] = pd.to_numeric(
    books_df["price_raw"]
        .str.replace("£", "", regex=False)
        .str.replace("Â", "", regex=False),
    errors="coerce"
)

In [14]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

books_df["rating_numeric"] = books_df["rating"].map(
    rating_map
)

In [15]:
books_df["missing_title"] = (
    books_df["title"].isna()
    | books_df["title"].str.strip().eq("")
)

books_df["invalid_price"] = (
    books_df["price_numeric"].isna()
    | books_df["price_numeric"].le(0)
)

books_df["invalid_rating"] = (
    books_df["rating_numeric"].isna()
)

books_df["duplicate_record"] = books_df.duplicated(
    subset=["title"],
    keep=False
)

In [17]:
def get_rejection_reason(row):
    reasons = []

    if row["missing_title"]:
        reasons.append("Missing title")

    if row["invalid_price"]:
        reasons.append("Invalid price")

    if row["invalid_rating"]:
        reasons.append("Invalid rating")

    if row["duplicate_record"]:
        reasons.append("Duplicate title")

    return "; ".join(reasons)


books_df["rejection_reason"] = books_df.apply(
    get_rejection_reason,
    axis=1
)

In [18]:
accepted_books = books_df[
    books_df["rejection_reason"] == ""
].copy()

rejected_books = books_df[
    books_df["rejection_reason"] != ""
].copy()

assert len(books_df) == (
    len(accepted_books) + len(rejected_books)
)

accepted_books.to_csv(
    "accepted_books.csv",
    index=False
)

rejected_books.to_csv(
    "rejected_books.csv",
    index=False
)

print("Scraped:", len(books_df))
print("Accepted:", len(accepted_books))
print("Rejected:", len(rejected_books))

Scraped: 100
Accepted: 100
Rejected: 0
